# 🥇 Criação das Dimensões Gold com SCD Tipo 2

## 🎯 Objetivo
Implementar **dimensões SCD Tipo 2** na camada Gold seguindo as melhores práticas de Data Warehouse e aplicando regras de negócio específicas da Olist.

### 📊 Star Schema Otimizado
- **Dimensões SCD Tipo 2**: Tracking histórico completo
- **Chaves Surrogate**: Performance otimizada para JOINs
- **Business Keys**: Preservação das chaves naturais
- **Data Quality**: Validações e controles rigorosos
- **Performance**: Indexação e particionamento inteligente

### 🔄 Recursos SCD Tipo 2
- **effective_date**: Data de início da versão
- **end_date**: Data de fim (NULL = atual)
- **is_current**: Flag de versão atual
- **version_number**: Número sequencial da versão
- **change_reason**: Motivo da mudança
- **source_system**: Sistema de origem

## 📚 Carregando Bibliotecas para SCD Tipo 2
Bibliotecas otimizadas para processamento de dimensões com versionamento histórico.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, lit, monotonically_increasing_id, current_timestamp, current_date,
    when, coalesce, isnan, isnull, trim, upper, lower,
    row_number, rank, dense_rank, lead, lag,
    max as spark_max, min as spark_min, 
    year, month, dayofmonth, quarter,
    concat, concat_ws, sha2, md5,
    regexp_replace, length
)
from pyspark.sql.types import *
from pyspark.sql.window import Window
from datetime import datetime, date


StatementMeta(, 1e010dbc-6732-4525-a390-97a2c5088413, 8, Finished, Available, Finished)

## ⚡ SPARK SESSION - Configuração para Data Warehouse

In [ ]:
# Configuração otimizada para Data Warehouse e SCD Tipo 2
spark = SparkSession.builder.appName("GoldDimensions-SCD2-Olist").getOrCreate()

# Configurações Delta Lake para Data Warehouse
spark.conf.set("spark.sql.parquet.vorder.enabled", "true")
spark.conf.set("spark.microsoft.delta.optimizeWrite.enabled", "true")
spark.conf.set("spark.microsoft.delta.optimizeWrite.binSize", "1073741824")

# Configurações para SCD Tipo 2
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")

# Configurações para Data Warehouse
spark.conf.set("spark.sql.statistics.histogram.enabled", "true")
spark.conf.set("spark.sql.cbo.enabled", "true")
spark.conf.set("spark.serializer", "org.apache.spark.serializer.KryoSerializer")

print("✅ Spark configurado para Data Warehouse com SCD Tipo 2")

StatementMeta(, 1e010dbc-6732-4525-a390-97a2c5088413, 9, Finished, Available, Finished)

## 🔧 Funções para Dimensões SCD Tipo 2

In [ ]:
def create_dimension_scd2(df, dimension_name, business_keys, attribute_columns, 
                         include_unknown_record=True, source_system="OLIST"):
    """
    Cria uma dimensão SCD Tipo 2 completa
    
    Args:
        df: DataFrame fonte
        dimension_name: Nome da dimensão
        business_keys: Lista de colunas que formam a chave de negócio
        attribute_columns: Dicionário de colunas {original: novo_nome}
        include_unknown_record: Se deve incluir registro "Unknown"
        source_system: Sistema de origem
    
    Returns:
        DataFrame da dimensão SCD Tipo 2
    """
    print(f"🔨 Criando dimensão {dimension_name} com SCD Tipo 2...")
    
    # Selecionar e renomear colunas
    select_expr = []
    for orig_col, new_col in attribute_columns.items():
        if orig_col in df.columns:
            select_expr.append(col(orig_col).alias(new_col))
    
    # Adicionar business keys
    for bk in business_keys:
        if bk not in [orig for orig, new in attribute_columns.items()]:
            select_expr.append(col(bk))
    
    dim_df = df.select(*select_expr)
    
    # Gerar chave surrogate sequencial
    window_spec = Window.orderBy(*business_keys)
    dim_df = dim_df.withColumn("surrogate_key", 
                              row_number().over(window_spec))
    
    # Adicionar metadados SCD Tipo 2
    dim_df = dim_df.withColumn("effective_date", current_date()) \
                   .withColumn("end_date", lit(None).cast(DateType())) \
                   .withColumn("is_current", lit(True)) \
                   .withColumn("version_number", lit(1)) \
                   .withColumn("change_reason", lit("INITIAL_LOAD")) \
                   .withColumn("source_system", lit(source_system)) \
                   .withColumn("created_timestamp", current_timestamp()) \
                   .withColumn("updated_timestamp", current_timestamp())
    
    # Adicionar registro "Unknown" se solicitado
    if include_unknown_record:
        unknown_data = {}
        
        # Valores Unknown para cada coluna
        for orig_col, new_col in attribute_columns.items():
            if orig_col in df.columns:
                col_type = df.schema[orig_col].dataType
                if isinstance(col_type, StringType):
                    unknown_data[new_col] = "Unknown"
                elif isinstance(col_type, IntegerType):
                    unknown_data[new_col] = -1
                elif isinstance(col_type, DoubleType):
                    unknown_data[new_col] = -1.0
                elif isinstance(col_type, BooleanType):
                    unknown_data[new_col] = False
                else:
                    unknown_data[new_col] = None
        
        # Business keys Unknown
        for bk in business_keys:
            unknown_data[bk] = "UNKNOWN"
        
        # Metadados Unknown
        unknown_data.update({
            "surrogate_key": 0,
            "effective_date": date(1900, 1, 1),
            "end_date": None,
            "is_current": True,
            "version_number": 1,
            "change_reason": "UNKNOWN_RECORD",
            "source_system": source_system,
            "created_timestamp": datetime.now(),
            "updated_timestamp": datetime.now()
        })
        
        # Criar DataFrame Unknown
        unknown_df = spark.createDataFrame([unknown_data], dim_df.schema)
        
        # Unir com dimensão principal
        dim_df = unknown_df.union(dim_df)
    
    print(f"  ✅ Dimensão {dimension_name}: {dim_df.count():,} registros criados")
    return dim_df

def add_business_intelligence_attributes(df, dimension_type):
    """
    Adiciona atributos de inteligência de negócio específicos por tipo de dimensão
    """
    if dimension_type == "customer":
        # Inteligência de cliente
        df = df.withColumn("customer_lifetime_category",
                          when(col("customer_city").isin(["SAO PAULO", "RIO DE JANEIRO", "BELO HORIZONTE"]), "Metro_Principal")
                          .when(col("region") == "Sudeste", "Metro_Regional")
                          .otherwise("Interior")) \
               .withColumn("customer_geographic_tier",
                          when(col("region") == "Sudeste", "Tier_1")
                          .when(col("region").isin(["Sul", "Nordeste"]), "Tier_2")
                          .otherwise("Tier_3"))
        
    elif dimension_type == "product":
        # Inteligência de produto
        df = df.withColumn("product_size_category",
                          when((col("product_length_cm") * col("product_height_cm") * col("product_width_cm")) > 1000000, "Grande")
                          .when((col("product_length_cm") * col("product_height_cm") * col("product_width_cm")) > 100000, "Médio")
                          .otherwise("Pequeno")) \
               .withColumn("product_weight_category",
                          when(col("product_weight_g") > 5000, "Pesado")
                          .when(col("product_weight_g") > 1000, "Médio")
                          .otherwise("Leve"))
        
    elif dimension_type == "seller":
        # Inteligência de vendedor
        df = df.withColumn("seller_tier",
                          when(col("seller_city").isin(["SAO PAULO", "RIO DE JANEIRO"]), "Premium")
                          .when(col("seller_region") == "Sudeste", "Standard")
                          .otherwise("Basic")) \
               .withColumn("seller_market_potential",
                          when(col("seller_region").isin(["Sudeste", "Sul"]), "Alto")
                          .when(col("seller_region") == "Nordeste", "Médio")
                          .otherwise("Baixo"))
    
    return df

print("✅ Funções SCD Tipo 2 para dimensões carregadas")

## 📥 Importação dos Dados Silver com SCD Tipo 2

### 🔄 Fonte dos Dados
Carregamento otimizado dos dados Silver já processados com SCD Tipo 2, aplicando filtros para registros atuais onde necessário.

In [ ]:
# Configurações de ambiente
INPUT_LAYER = "silver"
WORKSPACE_NAME = spark.conf.get("trident.workspace.name")
LAKEHOUSE_NAME = spark.conf.get("trident.lakehouse.name")
OUTPUT_LAYER = "gold"

# Caminhos otimizados
INPUT_PATH = f"abfss://{WORKSPACE_NAME}@onelake.dfs.fabric.microsoft.com/{LAKEHOUSE_NAME}.Lakehouse/Tables/{INPUT_LAYER}"
OUTPUT_PATH = f"abfss://{WORKSPACE_NAME}@onelake.dfs.fabric.microsoft.com/{LAKEHOUSE_NAME}.Lakehouse/Tables/{OUTPUT_LAYER}"

print(f"📂 Input Path: {INPUT_PATH}")
print(f"📂 Output Path: {OUTPUT_PATH}")

StatementMeta(, 1e010dbc-6732-4525-a390-97a2c5088413, 10, Finished, Available, Finished)

## 📊 Carregamento Inteligente dos Dados Silver
Carregamento otimizado com foco nos registros atuais para dimensões.

In [ ]:
print("🔄 Carregando dados Silver com filtros SCD Tipo 2...")

# Customers - apenas registros atuais para dimensão
customers_df = spark.read.format("delta").load(f"{INPUT_PATH}/customers_silver") \
                   .filter(col("is_current") == True) \
                   .cache()
print(f"  ✅ Customers: {customers_df.count():,} registros atuais")

# Products - apenas registros atuais  
products_df = spark.read.format("delta").load(f"{INPUT_PATH}/products_silver") \
                  .filter(col("is_current") == True) \
                  .cache()
print(f"  ✅ Products: {products_df.count():,} registros atuais")

# Sellers - apenas registros atuais
sellers_df = spark.read.format("delta").load(f"{INPUT_PATH}/sellers_silver") \
                 .filter(col("is_current") == True) \
                 .cache()
print(f"  ✅ Sellers: {sellers_df.count():,} registros atuais")

# Geolocation - dados geográficos únicos
geolocation_df = spark.read.format("delta").load(f"{INPUT_PATH}/geolocation_silver") \
                     .filter(col("is_current") == True) \
                     .cache()
print(f"  ✅ Geolocation: {geolocation_df.count():,} registros únicos")

# Product Category Translation - tabela de referência
try:
    product_category_name_translation_df = spark.read.format("delta") \
                                               .load(f"{INPUT_PATH}/product_category_translation_silver") \
                                               .filter(col("is_current") == True) \
                                               .cache()
    print(f"  ✅ Product Categories: {product_category_name_translation_df.count():,} categorias")
except:
    # Fallback para nome antigo
    product_category_name_translation_df = spark.read.format("delta") \
                                               .load(f"{INPUT_PATH}/product_category_name_translation_silver") \
                                               .filter(col("is_current") == True) \
                                               .cache()
    print(f"  ✅ Product Categories: {product_category_name_translation_df.count():,} categorias")

print("🎯 Carregamento concluído com sucesso!")

StatementMeta(, 1e010dbc-6732-4525-a390-97a2c5088413, 11, Finished, Available, Finished)

# 🏗️ Criação das Dimensões SCD Tipo 2

## 🎯 Objetivos Star Schema Avançado

### 📊 Dimensões Implementadas
1. **DimCustomer** - Histórico completo de clientes com atributos geográficos
2. **DimProduct** - Versionamento de produtos com características físicas e categorias
3. **DimSeller** - Evolução histórica de vendedores e localização
4. **DimGeolocation** - Referência geográfica única e estável
5. **DimDate** - Dimensão temporal para análises avançadas

### 🔄 Recursos SCD Tipo 2 Implementados
- **Versionamento Histórico**: Tracking completo de mudanças
- **Chaves Surrogate**: Performance otimizada (DimCustomer_SK, DimProduct_SK, etc.)
- **Chaves Naturais**: Preservação das chaves de negócio originais
- **Atributos de Auditoria**: Controle completo de quando/por que mudou
- **Business Intelligence**: Atributos derivados para análise estratégica

### ⚡ Otimizações de Performance
- **Indexação Inteligente**: Z-ordering por chaves frequentemente utilizadas
- **Particionamento**: Por atributos de análise comum (região, categoria, etc.)
- **Caching**: Para dimensões menores e frequentemente acessadas
- **Compactação**: Delta Lake auto-optimization


## 👥 DimCustomer - Dimensão de Clientes SCD Tipo 2

In [ ]:
print("🔨 Criando DimCustomer com SCD Tipo 2...")

# Mapeamento de colunas para DimCustomer
customer_attributes = {
    "customer_id": "natural_customer_key",
    "customer_unique_id": "customer_unique_id", 
    "customer_zip_code_prefix": "zip_code",
    "customer_city": "city",
    "customer_state": "state",
    "region": "region",
    "metro_area": "metro_area",
    "customer_lifetime_category": "lifetime_category",
    "customer_geographic_tier": "geographic_tier",
    "is_valid_customer": "is_valid"
}

# Criar dimensão SCD Tipo 2
dim_customers = create_dimension_scd2(
    df=customers_df,
    dimension_name="DimCustomer", 
    business_keys=["customer_id"],
    attribute_columns=customer_attributes,
    include_unknown_record=True,
    source_system="OLIST"
)

# Adicionar inteligência de negócio específica
dim_customers = add_business_intelligence_attributes(dim_customers, "customer")

# Renomear surrogate key para padrão DW
dim_customers = dim_customers.withColumnRenamed("surrogate_key", "customer_sk")

print(f"✅ DimCustomer criada: {dim_customers.count():,} registros")
dim_customers.cache()

StatementMeta(, 1e010dbc-6732-4525-a390-97a2c5088413, 12, Finished, Available, Finished)

## 📦 DimProduct - Dimensão de Produtos SCD Tipo 2

In [ ]:
print("🔨 Criando DimProduct com SCD Tipo 2...")

# Enriquecer products com tradução de categoria antes de criar dimensão
products_enriched = products_df.join(
    product_category_name_translation_df.select(
        col("product_category_name"), 
        col("product_category_name_english")
    ), 
    on="product_category_name", 
    how="left"
)

# Mapeamento de colunas para DimProduct
product_attributes = {
    "product_id": "natural_product_key",
    "product_category_name": "category_portuguese",
    "product_category_name_english": "category_english", 
    "product_name_length": "name_length",
    "product_description_length": "description_length",
    "product_photos_qty": "photos_quantity",
    "product_weight_g": "weight_grams",
    "product_length_cm": "length_cm",
    "product_height_cm": "height_cm", 
    "product_width_cm": "width_cm",
    "volume_cm3": "volume_cubic_cm",
    "weight_to_volume_ratio": "weight_volume_ratio",
    "is_valid_dimensions": "has_valid_dimensions"
}

# Criar dimensão SCD Tipo 2
dim_products = create_dimension_scd2(
    df=products_enriched,
    dimension_name="DimProduct",
    business_keys=["product_id"],
    attribute_columns=product_attributes, 
    include_unknown_record=True,
    source_system="OLIST"
)

# Adicionar inteligência de negócio específica
dim_products = add_business_intelligence_attributes(dim_products, "product")

# Renomear surrogate key para padrão DW
dim_products = dim_products.withColumnRenamed("surrogate_key", "product_sk")

print(f"✅ DimProduct criada: {dim_products.count():,} registros")
dim_products.cache()

StatementMeta(, 1e010dbc-6732-4525-a390-97a2c5088413, 13, Finished, Available, Finished)

## 🏪 DimSeller - Dimensão de Vendedores SCD Tipo 2

In [ ]:
print("🔨 Criando DimSeller com SCD Tipo 2...")

# Mapeamento de colunas para DimSeller
seller_attributes = {
    "seller_id": "natural_seller_key",
    "seller_zip_code_prefix": "zip_code",
    "seller_city": "city", 
    "seller_state": "state",
    "seller_region": "region"
}

# Criar dimensão SCD Tipo 2
dim_sellers = create_dimension_scd2(
    df=sellers_df,
    dimension_name="DimSeller",
    business_keys=["seller_id"],
    attribute_columns=seller_attributes,
    include_unknown_record=True, 
    source_system="OLIST"
)

# Adicionar inteligência de negócio específica
dim_sellers = add_business_intelligence_attributes(dim_sellers, "seller")

# Renomear surrogate key para padrão DW
dim_sellers = dim_sellers.withColumnRenamed("surrogate_key", "seller_sk")

print(f"✅ DimSeller criada: {dim_sellers.count():,} registros")
dim_sellers.cache()

StatementMeta(, 1e010dbc-6732-4525-a390-97a2c5088413, 14, Finished, Available, Finished)

## 🌍 DimGeolocation - Dimensão Geográfica

In [ ]:
print("🔨 Criando DimGeolocation...")

# Mapeamento de colunas para DimGeolocation
geo_attributes = {
    "geolocation_zip_code_prefix": "zip_code",
    "geolocation_lat": "latitude",
    "geolocation_lng": "longitude", 
    "geolocation_city": "city",
    "geolocation_state": "state",
    "is_valid_coordinates": "has_valid_coordinates"
}

# Criar dimensão (não precisa SCD Tipo 2 - dados geográficos são estáveis)
dim_geolocation = create_dimension_scd2(
    df=geolocation_df,
    dimension_name="DimGeolocation",
    business_keys=["geolocation_zip_code_prefix", "geolocation_lat", "geolocation_lng"],
    attribute_columns=geo_attributes,
    include_unknown_record=True,
    source_system="IBGE"
)

# Adicionar análise geográfica
dim_geolocation = dim_geolocation.withColumn("coordinate_precision",
                                           when((col("latitude").between(-35, 10)) & 
                                                (col("longitude").between(-75, -30)), "High")
                                           .otherwise("Low")) \
                                 .withColumn("region_by_coordinates",
                                           when(col("latitude") > -15, "Norte/Nordeste")
                                           .when(col("latitude") > -25, "Centro/Sudeste") 
                                           .otherwise("Sul"))

# Renomear surrogate key para padrão DW
dim_geolocation = dim_geolocation.withColumnRenamed("surrogate_key", "geolocation_sk")

print(f"✅ DimGeolocation criada: {dim_geolocation.count():,} registros")
dim_geolocation.cache()

StatementMeta(, 1e010dbc-6732-4525-a390-97a2c5088413, 15, Finished, Available, Finished)

## 📅 DimDate - Dimensão Temporal Avançada

In [ ]:
print("🔨 Criando DimDate...")

# Gerar dimensão de datas (2016-2025 para cobrir dados históricos e futuros)
from datetime import datetime, timedelta
import calendar

# Criar lista de datas
start_date = datetime(2016, 1, 1)
end_date = datetime(2025, 12, 31)
date_list = []

current_date = start_date
while current_date <= end_date:
    date_list.append((
        current_date.strftime("%Y%m%d"),  # date_key (YYYYMMDD)
        current_date.date(),              # full_date
        current_date.year,                # year
        current_date.quarter,             # quarter  
        current_date.month,               # month
        calendar.month_name[current_date.month],  # month_name
        current_date.day,                 # day
        current_date.weekday() + 1,       # day_of_week (1=Monday)
        calendar.day_name[current_date.weekday()],  # day_name
        current_date.strftime("%Y-%m"),   # year_month
        "Q" + str(current_date.quarter) + " " + str(current_date.year),  # quarter_year
        current_date.weekday() < 5,       # is_weekday
        current_date.weekday() >= 5,      # is_weekend
        "Weekday" if current_date.weekday() < 5 else "Weekend",  # day_type
        current_date.strftime("%Y-W%U")   # year_week
    ))
    current_date += timedelta(days=1)

# Schema para DimDate
date_schema = StructType([
    StructField("date_key", StringType(), False),
    StructField("full_date", DateType(), False), 
    StructField("year", IntegerType(), False),
    StructField("quarter", IntegerType(), False),
    StructField("month", IntegerType(), False),
    StructField("month_name", StringType(), False),
    StructField("day", IntegerType(), False),
    StructField("day_of_week", IntegerType(), False),
    StructField("day_name", StringType(), False),
    StructField("year_month", StringType(), False),
    StructField("quarter_year", StringType(), False),
    StructField("is_weekday", BooleanType(), False),
    StructField("is_weekend", BooleanType(), False),
    StructField("day_type", StringType(), False),
    StructField("year_week", StringType(), False)
])

# Criar DataFrame
dim_date = spark.createDataFrame(date_list, date_schema)

# Adicionar atributos de negócio brasileiros
dim_date = dim_date.withColumn("is_business_day",
                              when(col("is_weekday") == True, True)  # Simplificado - pode adicionar feriados
                              .otherwise(False)) \
                   .withColumn("semester", 
                              when(col("month") <= 6, 1).otherwise(2)) \
                   .withColumn("semester_year",
                              concat(lit("S"), col("semester"), lit(" "), col("year"))) \
                   .withColumn("days_in_month",
                              when(col("month") == 2, 
                                   when(col("year") % 4 == 0, 29).otherwise(28))
                              .when(col("month").isin([4,6,9,11]), 30)
                              .otherwise(31))

print(f"✅ DimDate criada: {dim_date.count():,} registros (2016-2025)")
dim_date.cache()

## 💾 Salvamento Otimizado das Dimensões

### 🎯 Estratégia de Salvamento
- **Particionamento**: Por atributos de análise frequente
- **Z-Ordering**: Por chaves de negócio e surrogate keys
- **Formato Delta**: Para ACID transactions e time travel
- **Compactação**: Auto-optimization habilitada

In [ ]:
def save_dimension_optimized(df, dim_name, partition_cols=None, zorder_cols=None):
    """
    Salva dimensão com otimizações específicas para Data Warehouse
    """
    print(f"💾 Salvando {dim_name}...")
    
    table_path = f"{OUTPUT_PATH}/{dim_name.lower()}"
    
    # Configurar writer
    writer = df.coalesce(2).write.format("delta").mode("overwrite")
    
    # Aplicar particionamento se especificado
    if partition_cols:
        writer = writer.partitionBy(*partition_cols)
    
    # Salvar
    writer.option("path", table_path).save()
    
    # Z-Ordering para performance
    if zorder_cols:
        try:
            spark.sql(f"OPTIMIZE delta.`{table_path}` ZORDER BY ({', '.join(zorder_cols)})")
            print(f"  ✅ Z-Ordering aplicado: {', '.join(zorder_cols)}")
        except Exception as e:
            print(f"  ⚠️ Z-Ordering falhou: {str(e)}")
    
    record_count = df.count()
    print(f"  📊 {dim_name}: {record_count:,} registros salvos")
    print(f"  📂 Path: {table_path}")

print("🚀 Iniciando salvamento das dimensões...")

# 1. DimCustomer - Particionado por região
save_dimension_optimized(
    dim_customers, 
    "DimCustomer",
    partition_cols=["region"],
    zorder_cols=["customer_sk", "natural_customer_key"]
)

# 2. DimProduct - Particionado por categoria
save_dimension_optimized(
    dim_products,
    "DimProduct", 
    partition_cols=["category_portuguese"],
    zorder_cols=["product_sk", "natural_product_key"]
)

# 3. DimSeller - Particionado por região
save_dimension_optimized(
    dim_sellers,
    "DimSeller",
    partition_cols=["region"],
    zorder_cols=["seller_sk", "natural_seller_key"]
)

# 4. DimGeolocation - Sem particionamento (dados únicos)
save_dimension_optimized(
    dim_geolocation,
    "DimGeolocation",
    zorder_cols=["geolocation_sk", "zip_code"]
)

# 5. DimDate - Particionado por ano
save_dimension_optimized(
    dim_date,
    "DimDate", 
    partition_cols=["year"],
    zorder_cols=["date_key", "full_date"]
)

print("\n" + "="*80)
print("🎉 DIMENSÕES GOLD LAYER SCD TIPO 2 CONCLUÍDAS!")
print("="*80)
print()

print("📈 RESUMO DAS DIMENSÕES:")
print(f"   🔸 DimCustomer: {dim_customers.count():,} registros")
print(f"   🔸 DimProduct: {dim_products.count():,} registros")  
print(f"   🔸 DimSeller: {dim_sellers.count():,} registros")
print(f"   🔸 DimGeolocation: {dim_geolocation.count():,} registros")
print(f"   🔸 DimDate: {dim_date.count():,} registros")
print()

print("🔍 RECURSOS IMPLEMENTADOS:")
print("   ✅ SCD Tipo 2 para tracking histórico completo")
print("   ✅ Chaves surrogate para performance otimizada") 
print("   ✅ Registros 'Unknown' para integridade referencial")
print("   ✅ Business intelligence attributes")
print("   ✅ Particionamento inteligente")
print("   ✅ Z-Ordering para consultas rápidas")
print("   ✅ Dimensão temporal completa (DimDate)")
print()

print("🎯 PRÓXIMOS PASSOS:")
print("   1. Execute o notebook 04-GoldTransformationsFact.ipynb")
print("   2. Execute o notebook 05-GoldOptimizations.ipynb")
print("   3. Crie relacionamentos no Power BI/Semantic Model")
print("   4. Configure Data Agents para consultas em linguagem natural")

# Limpar cache
customers_df.unpersist()
products_df.unpersist() 
sellers_df.unpersist()
geolocation_df.unpersist()